In [1]:
import sys
from pathlib import Path
 
print("Python:", sys.version.split()[0])
print("Folder:", Path.cwd().name)
 
for name in ["numpy", "pandas", "sklearn"]:
    try:
        __import__(name)
        print(name, "- ok")
    except ImportError:
        print(name, "- missing")

Python: 3.10.19
Folder: MLOPS lab
numpy - ok
pandas - ok
sklearn - ok


# Build the dataset

In [2]:
import csv
from pathlib import Path
import numpy as np
 
SEED = 42
N_ROWS = 600
DATA = Path("data") / "delivery_times.csv"

def make_delivery_csv(path=DATA):
    rng = np.random.default_rng(SEED)
    distance_km = np.round(rng.uniform(0.5, 12.0, N_ROWS), 2)
    prep_time_min = np.round(rng.uniform(5, 30, N_ROWS), 0)
    traffic_level = rng.integers(1, 4, N_ROWS)
    rain = rng.binomial(1, 0.25, N_ROWS)
    delivery_min = np.round(
        6.0 + 3.1 * distance_km + 0.65 * prep_time_min
        + 4.2 * traffic_level + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS), 1)
 
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["distance_km", "prep_time_min", "traffic_level", "rain", "delivery_min"])
        for i in range(N_ROWS):
            w.writerow([distance_km[i], int(prep_time_min[i]), int(traffic_level[i]), int(rain[i]), delivery_min[i]])
    return path
 
 
if not DATA.exists():
    make_delivery_csv()
print("dataset ready:", DATA)


dataset ready: data/delivery_times.csv


# Loads the data, separates features (X) from the target (y), and does an 80/20 train/test split.

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
 
orders = pd.read_csv(DATA)
FEATURES = ["distance_km", "prep_time_min", "traffic_level", "rain"]
X = orders[FEATURES]
y = orders["delivery_min"]
 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(len(X_train), len(X_test))

480 120


# Define a function that trains a model, measures its error on both the training data and the test data, and computes the gap between them.

In [4]:
from sklearn.metrics import mean_absolute_error
 
def score_both_ways(model, name):
    model.fit(X_train, y_train)
    train_mae = mean_absolute_error(y_train, model.predict(X_train))
    test_mae = mean_absolute_error(y_test, model.predict(X_test))
    gap = test_mae - train_mae
    print(name, "train", round(train_mae, 2), "test", round(test_mae, 2), "gap", round(gap, 2))
    return {"name": name, "train": train_mae, "test": test_mae, "gap": gap}

# Model 1: LinearRegression

In [5]:
from sklearn.linear_model import LinearRegression
 
linear = score_both_ways(LinearRegression(), "LinearRegression")

LinearRegression train 2.04 test 1.92 gap -0.11


# Model 2: decision tree, no limit

In [6]:
from sklearn.tree import DecisionTreeRegressor
 
wild_tree = score_both_ways(DecisionTreeRegressor(random_state=42), "DecisionTree (no limit)")

DecisionTree (no limit) train 0.0 test 3.43 gap 3.43


# Model 3: shallow tree (depth 4)

In [7]:
small_tree = score_both_ways(DecisionTreeRegressor(max_depth=4, random_state=42), "DecisionTree (depth 4)")

DecisionTree (depth 4) train 3.66 test 4.23 gap 0.57


# Model 4: RandomForest

In [8]:
from sklearn.ensemble import RandomForestRegressor
 
forest = score_both_ways(RandomForestRegressor(n_estimators=50, random_state=42), "RandomForest (50 trees)")

RandomForest (50 trees) train 0.99 test 2.4 gap 1.4


In [9]:
results = pd.DataFrame([linear, wild_tree, small_tree, forest]).round(2).sort_values("test")
print(results.to_string(index=False))


                   name  train  test   gap
       LinearRegression   2.04  1.92 -0.11
RandomForest (50 trees)   0.99  2.40  1.40
DecisionTree (no limit)   0.00  3.43  3.43
 DecisionTree (depth 4)   3.66  4.23  0.57


# Cross-validation - Trains and tests each model 5 times on 5 different data splits and averages the error

In [10]:
from sklearn.model_selection import cross_val_score
 
def cross_validate(model, name):
    scores = -cross_val_score(model, X, y, cv=5, scoring="neg_mean_absolute_error")
    print(name, "MAE", round(scores.mean(), 2))
    return scores.mean()
 
cv_linear = cross_validate(LinearRegression(), "LinearRegression")
cv_tree = cross_validate(DecisionTreeRegressor(max_depth=4, random_state=42), "DecisionTree (depth 4)")
cv_forest = cross_validate(RandomForestRegressor(n_estimators=50, random_state=42), "RandomForest (50 trees)")

LinearRegression MAE 2.03


DecisionTree (depth 4) MAE 4.57
RandomForest (50 trees) MAE 2.69


# Tries several tree depths (2, 3, 4, 6, 8, None), scores each with cross-validation, and finds the best one.

In [11]:
depths = [2, 3, 4, 6, 8, None]
scores_by_depth = {}
 
for depth in depths:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=42)
    scores = cross_val_score(tree, X, y, cv=5,
                             scoring="neg_mean_absolute_error")
    mae = -scores.mean()
    scores_by_depth[depth] = round(float(mae), 3)
 
best_depth = min(scores_by_depth, key=scores_by_depth.get)
print(scores_by_depth)
print("best depth:", best_depth)

{2: 5.858, 3: 4.85, 4: 4.573, 6: 3.642, 8: 3.395, None: 3.473}
best depth: 8


# Ranks the three model

In [12]:
ranking = sorted(
    {"LinearRegression": cv_linear, "DecisionTree(4)": cv_tree, "RandomForest(50)": cv_forest}.items(),
    key=lambda kv: kv[1],
)
for name, mae in ranking:
    print(name, round(mae, 2))

LinearRegression 2.03
RandomForest(50) 2.69
DecisionTree(4) 4.57


# To Do: Train Logistic Regression, Decision Tree, and Random Forest classifiers on the Breast Cancer dataset, compare their train/test accuracy, use cross-validation to find the best tree depth, and report the confusion matrix and classification metrics to identify the best-performing model. https://www.kaggle.com/datasets/yasserh/breast-cancer-dataset

In [42]:
# Loads the Breast Cancer dataset and separates features (X) from target (y)

import pandas as pd
from pathlib import Path

DATA = Path("data") / "breast_cancer.csv"

cancer = pd.read_csv(DATA)

print("dataset shape:", cancer.shape)
print(cancer.head())

dataset shape: (569, 32)
         id diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0    842302         M        17.99         10.38          122.80     1001.0   
1    842517         M        20.57         17.77          132.90     1326.0   
2  84300903         M        19.69         21.25          130.00     1203.0   
3  84348301         M        11.42         20.38           77.58      386.1   
4  84358402         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   ...  radius_worst  texture_worst  

In [43]:
# Separates features (X) from target (y) and does an 80/20 train/test split

from sklearn.model_selection import train_test_split

cancer = cancer.drop(columns=["id", "Unnamed: 32"], errors="ignore")

cancer["diagnosis"] = cancer["diagnosis"].map({"M": 1, "B": 0})

X = cancer.drop(columns=["diagnosis"])
y = cancer["diagnosis"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(len(X_train), len(X_test))

455 114


In [44]:
# Defines a function that trains a model and measures its train and test accuracy

from sklearn.metrics import accuracy_score

def score_both_ways(model, name):
    model.fit(X_train, y_train)

    train_accuracy = accuracy_score(
        y_train, model.predict(X_train)
    )

    test_accuracy = accuracy_score(
        y_test, model.predict(X_test)
    )

    gap = train_accuracy - test_accuracy

    print(
        name,
        "train", round(train_accuracy, 4),
        "test", round(test_accuracy, 4),
        "gap", round(gap, 4)
    )

    return {
        "name": name,
        "train": train_accuracy,
        "test": test_accuracy,
        "gap": gap
    }

In [45]:
# Model 1: Logistic Regression

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logistic_model = LogisticRegression(max_iter=1000, random_state=42)

logistic_model.fit(X_train_scaled, y_train)

logistic_train = accuracy_score(
    y_train, logistic_model.predict(X_train_scaled)
)

logistic_test = accuracy_score(
    y_test, logistic_model.predict(X_test_scaled)
)

logistic = {
    "name": "LogisticRegression",
    "train": logistic_train,
    "test": logistic_test,
    "gap": logistic_train - logistic_test
}

print(
    "LogisticRegression",
    "train", round(logistic_train, 4),
    "test", round(logistic_test, 4),
    "gap", round(logistic["gap"], 4)
)

LogisticRegression train 0.9868 test 0.9737 gap 0.0131


In [46]:
# Model 2: Decision Tree

from sklearn.tree import DecisionTreeClassifier

tree = score_both_ways(
    DecisionTreeClassifier(random_state=42),
    "DecisionTree"
)

DecisionTree train 1.0 test 0.9474 gap 0.0526


In [47]:
# Model 3: Random Forest

from sklearn.ensemble import RandomForestClassifier

forest = score_both_ways(
    RandomForestClassifier(n_estimators=100, random_state=42),
    "RandomForest"
)

RandomForest train 1.0 test 0.9649 gap 0.0351


In [48]:
# Compares the train and test accuracy of all three models

results = pd.DataFrame(
    [logistic, tree, forest]
).round(4).sort_values("test", ascending=False)

print(results.to_string(index=False))

              name  train   test    gap
LogisticRegression 0.9868 0.9737 0.0131
      RandomForest 1.0000 0.9649 0.0351
      DecisionTree 1.0000 0.9474 0.0526


In [49]:
# Tries several tree depths using cross-validation and finds the best one

from sklearn.model_selection import cross_val_score

depths = [2, 3, 4, 5, 6, 8, 10, None]
scores_by_depth = {}

for depth in depths:
    tree_model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )

    scores = cross_val_score(
        tree_model,
        X,
        y,
        cv=5,
        scoring="accuracy"
    )

    scores_by_depth[depth] = round(scores.mean(), 4)

best_depth = max(
    scores_by_depth,
    key=scores_by_depth.get
)

print(scores_by_depth)
print("best depth:", best_depth)

{2: 0.928, 3: 0.9191, 4: 0.9209, 5: 0.9191, 6: 0.9209, 8: 0.9156, 10: 0.9173, None: 0.9173}
best depth: 2


In [50]:
# Trains a Decision Tree using the best depth found by cross-validation

best_tree = score_both_ways(
    DecisionTreeClassifier(
        max_depth=best_depth,
        random_state=42
    ),
    f"DecisionTree (depth {best_depth})"
)

DecisionTree (depth 2) train 0.9297 test 0.9298 gap -0.0002


In [51]:
# Reports the confusion matrix and classification metrics

from sklearn.metrics import confusion_matrix, classification_report

models = {
    "LogisticRegression": logistic_model,
    "DecisionTree": DecisionTreeClassifier(
        max_depth=best_depth,
        random_state=42
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
}

for name, model in models.items():

    if name == "LogisticRegression":
        predictions = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

    print("\n", "=" * 40)
    print(name)
    print("=" * 40)

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, predictions))

    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            predictions,
            target_names=["Benign", "Malignant"]
        )
    )


LogisticRegression
Confusion Matrix:
[[70  1]
 [ 2 41]]

Classification Report:
              precision    recall  f1-score   support

      Benign       0.97      0.99      0.98        71
   Malignant       0.98      0.95      0.96        43

    accuracy                           0.97       114
   macro avg       0.97      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114


DecisionTree
Confusion Matrix:
[[69  2]
 [ 6 37]]

Classification Report:
              precision    recall  f1-score   support

      Benign       0.92      0.97      0.95        71
   Malignant       0.95      0.86      0.90        43

    accuracy                           0.93       114
   macro avg       0.93      0.92      0.92       114
weighted avg       0.93      0.93      0.93       114


RandomForest
Confusion Matrix:
[[70  1]
 [ 3 40]]

Classification Report:
              precision    recall  f1-score   support

      Benign       0.96      0.99      0.97        71
   Mal

In [52]:
# Ranks the three models according to their test accuracy

final_results = pd.DataFrame([
    logistic,
    tree,
    forest
])

final_results = final_results.sort_values(
    "test",
    ascending=False
)

print(final_results.round(4).to_string(index=False))

best_model = final_results.iloc[0]

print("\nBest-performing model:", best_model["name"])
print("Test accuracy:", round(best_model["test"], 4))

              name  train   test    gap
LogisticRegression 0.9868 0.9737 0.0131
      RandomForest 1.0000 0.9649 0.0351
      DecisionTree 1.0000 0.9474 0.0526

Best-performing model: LogisticRegression
Test accuracy: 0.9737
